# Identify and download co-located data from NEON and EMIT:
* NEON:
** Identify burned and unburned tiles of interest from the NEON AOP data. Identify the latitude and longitude of those tiles.
Download the NEON Spectrometer orthorectified surface bidirectional reflectance data for the burned and unburned tiles.
* EMIT:
** Identify EMIT L2A Estimated Surface Reflectance granule(s) that cover the burned and unburned tiles.
** Download the EMIT L2A Estimated Surface Reflectance granule(s)
** Clip/crop the downloaded EMIT L2A Estimated Surface Reflectance granule(s) to the burned and unburned tiles of interest.

# Notes:
The CRS is EPSG:4326 (WGS84), which is also the CRS we want the data in to submit for our search of EMIT data.


This notebook will be used to find and download data around the National Ecological Observatory Network (NEON) Soaproot Saddle (SOAP) field site. The SOAP site is in the Sierra National Forest in California and the EMIT data we use in this notebook is from July 31, 2023. 

The data calculations in the next notebook will be done using the Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product. This notebook is inspired by an existing notebook created by the Land Processes Distributed Active Archive Center (LP DAAC). The existing notebook is called 3 Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data. 

EMIT L2A Dataset Citation:

Green, R. (2022). EMIT L2A Estimated Surface Reflectance and Uncertainty and Masks 60 m V001 [Data set]. NASA Land Processes Distributed Active Archive Center. https://doi.org/10.5067/EMIT/EMITL2ARFL.001 Date Accessed: 2025-06-19

Notebooks used: 
* 01_bh_find_collocated_neon_emit_data
* 04_rn_download_neon_l3_shapefiles


1.0 Setup

In [6]:
# Import required libraries
import os, sys
import requests
import folium
import earthaccess
import warnings
import csv
import folium.plugins
import pandas as pd
import geopandas as gpd
import numpy as np
import math
import rasterio as rio
from rasterio.plot import show, show_hist
import requests
import xarray as xr
import holoviews as hv
import hvplot.xarray
import netCDF4 as nc

from zipfile import ZipFile
from branca.element import Figure
from IPython.display import display
from shapely import geometry
from skimage import io
from datetime import timedelta
from shapely.geometry.polygon import orient
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from osgeo import gdal

# This will ignore some warnings caused by holoviews
warnings.simplefilter('ignore') 

If not already installed, install the neonutilities and python-dotenv packages using pip as follows:
!pip install neonutilities
!pip install python-dotenv

In [2]:
import neonutilities as nu
import dotenv

Login to your NASA Earthdata account and 
create a .netrc file using the login function from the earthaccess library. 
If you do not have an Earthdata Account, you can create one here.

In [ ]:
earthaccess.login(persist=True)

For this notebook we will download the files necessary using earthaccess. You can also access the data in place or stream it, but this can slow due to the file sizes. Provide a URL for an EMIT L2A Reflectance granule.

In [ ]:
url = 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20220903T163129_2224611_012/EMIT_L2A_RFL_001_20220903T163129_2224611_012.nc'

Get an HTTPS Session using your earthdata login, set a local path to save the file, and download the granule asset - This may take a while, the reflectance file is approximately 1.8 GB.

In [ ]:
# Get requests https Session using Earthdata Login Info
fs = earthaccess.get_requests_https_session()
# Retrieve granule asset ID from URL (to maintain existing naming convention)
granule_asset_id = url.split('/')[-1]
# Define Local Filepath
fp = f'../../data/{granule_asset_id}'
# Download the Granule Asset if it doesn't exist
if not os.path.isfile(fp):
    with fs.get(url,stream=True) as src:
        with open(fp,'wb') as dst:
            for chunk in src.iter_content(chunk_size=64*1024*1024):
                dst.write(chunk)

1.2 NEON Data Exploration and Download
In this section we will load a shapefile of the NEON site boundaries, take a look at the SOAP site, and then find and download SOAP reflectance data. 
First, let's define a function that will download data from a url. We will use this to download a shapefile boundary of the NEON AOP flight boxes.

In [3]:
# function to download data stored on the internet in a public url to a local file
def download_url(url,download_dir):
    if not os.path.isdir(download_dir):
        os.makedirs(download_dir)
    filename = url.split('/')[-1]
    r = requests.get(url, allow_redirects=True)
    file_object = open(os.path.join(download_dir,filename),'wb')
    file_object.write(r.content)

2.0 Search for NEON and EMIT Data
2.1 Define Spatial Regions of Interest (ROIs)

In [4]:
# Download and Unzip the NEON Flight Boundary Shapefile
neon_boundary_url = "https://www.neonscience.org/sites/default/files/AOP_flightBoxes_0.zip"
# Use download_url function to save the file to a directory
os.makedirs('./data', exist_ok=True)
download_url(neon_boundary_url,'./data')
# Unzip the file
with ZipFile(f"./data/{neon_boundary_url.split('/')[-1]}", 'r') as zip_ref:
    zip_ref.extractall('./data')

In [5]:
aop_flightboxes = gpd.read_file("./data/AOP_flightBoxes/AOP_flightboxesAllSites.shp")
aop_flightboxes.head()

,domain,domainName,siteName,siteID,siteType,sampleType,priority,version,flightbxID,geometry
0,D01,Northeast,Bartlett Experimental Forest NEON,BART,Gradient,Terrestrial,1,1,D01_BART_R1_P1_v1,"POLYGON ((-71.33426 43.99197, -71.33423 44.081..."
1,D01,Northeast,Harvard Forest & Quabbin Watershed NEON,HARV,Core,Terrestrial,1,1,D01_HARV_C1_P1_v1,"POLYGON ((-72.14819 42.5751, -72.14776 42.3837..."
2,D01,Northeast,Harvard Forest & Quabbin Watershed NEON,HARV,Core,Terrestrial,3,1,D01_HARV_C1_P3_v1,"POLYGON ((-72.10812 42.43653, -72.14788 42.436..."
3,D01,Northeast,Lower Hop Brook NEON,HOPB,Core,Aquatic,2,1,D01_HOPB_C1_P2_v1,"POLYGON ((-72.36635 42.46399, -72.36635 42.514..."
4,D19,Taiga,Healy NEON,HEAL,Gradient,Terrestrial,1,1,D19_HEAL_R3_P1_v1,"POLYGON ((-149.31505 63.82981, -149.31505 63.9..."


In [7]:
site_id = 'SOAP'
aop_flightboxes[aop_flightboxes.siteID == site_id]

,domain,domainName,siteName,siteID,siteType,sampleType,priority,version,flightbxID,geometry
89,D17,Pacific Southwest,Soaproot Saddle NEON,SOAP,Gradient,Terrestrial,1,6,D17_SOAP_R1_P1_v6,"POLYGON ((-119.29195 37.0143, -119.29192 37.10..."
90,D17,Pacific Southwest,Soaproot Saddle NEON,SOAP,Gradient,Terrestrial,2,2,D17_SOAP_R1_P2_v2,"POLYGON ((-119.32506 37.01417, -119.32506 37.0..."
91,D17,Pacific Southwest,Soaproot Saddle NEON,SOAP,Gradient,Terrestrial,3,1,D17_SOAP_R1_P3_v1,"POLYGON ((-119.31839 36.98832, -119.31839 37.0..."


Question: Do we want to go the shapefile route (notebook 01_bh_find_collocated_neon_emit_data 
or do we want to go the CHM route (notebook 04_rn_download_neon_l3_shapefiles

Downloading EMIT data - uses notebook 03a_rnExploring_EMIT_L2A_Reflectance

3.0 Opening EMIT Data

In [ ]:
ds_nc = nc.Dataset(fp)
ds_nc

In [ ]:
ds_nc['location']